# Inspect and combine legacy benchmark shards

This notebook treats `outputs/results/shards/splits.pkl` as the source of truth, inspects every expected `classical.pkl` and `gcn.pkl` shard, validates their runs against the saved splits, and writes one standard benchmark artifact. Existing shard files are never modified.

The legacy GCN shards contain the same genes as the classical shards but in a different node order. The merge therefore aligns score arrays by gene ID before combining them.

In [1]:
from __future__ import annotations

import os
import pickle
import sys
from pathlib import Path

import numpy as np
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'cluster' / 'sim.py').is_file() and (candidate / 'outputs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the bioGraph project root.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SHARD_ROOT = PROJECT_ROOT / 'outputs' / 'results' / 'shards'
MANIFEST_PATH = SHARD_ROOT / 'splits.pkl'
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'results' / 'results_methods_all_shards.pkl'

print(f'Project:  {PROJECT_ROOT}')
print(f'Manifest: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}')
print(f'Output:   {OUTPUT_PATH.relative_to(PROJECT_ROOT)}')

Project:  /Users/valentin/Documents/1_Promotion/Biological Networks/bioGraph
Manifest: outputs/results/shards/splits.pkl
Output:   outputs/results/results_methods_all_shards.pkl


## Load compatibility

These old files were produced with a NumPy version that pickled arrays via `numpy._core`. NumPy 1.x exposes the same implementation as `numpy.core`, so the aliases below allow trusted, locally produced legacy pickles to load. Pickle files can execute code while loading; do not use this notebook on untrusted files.

In [2]:
# Compatibility for shards created by NumPy 2.x and read with NumPy 1.x.
sys.modules.setdefault('numpy._core', np.core)
sys.modules.setdefault('numpy._core.numeric', np.core.numeric)

# Kept local so the legacy-data notebook does not import the full simulation stack.
ARTIFACT_SCHEMA_VERSION = 8
CLUSTER_SCHEMA_VERSION = 1
METHOD_GROUPS = ('classical', 'gcn')


def validate_benchmark_results(result: dict) -> None:
    assert isinstance(result, dict), 'Artifact must be a dictionary.'
    assert result.get('schema_version') == ARTIFACT_SCHEMA_VERSION, 'Unsupported artifact schema.'
    assert {'config', 'nodelist', 'runs'} <= set(result), 'Artifact keys are incomplete.'
    config = result['config']
    assert {'disease_set', 'method_set', 'num_runs', 'split_fraction', 'base_seed', 'hyperparameters'} <= set(config)
    assert isinstance(result['nodelist'], list) and result['nodelist'], 'Nodelist is empty.'
    assert isinstance(result['runs'], list), 'Runs must be a list.'
    for run in result['runs']:
        assert {'disease', 'seed', 'train_genes', 'test_genes', 'scores'} <= set(run)
        assert set(run['scores']) == set(config['method_set'])
        assert all(np.asarray(score).shape == (len(result['nodelist']),) for score in run['scores'].values())


def load_pickle(path: Path):
    with path.open('rb') as handle:
        return pickle.load(handle)


def atomic_pickle_dump(value, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f'.{destination.name}.{os.getpid()}.tmp')
    try:
        with temporary.open('wb') as handle:
            pickle.dump(value, handle, protocol=pickle.HIGHEST_PROTOCOL)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)

## Manifest and shard inventory

In [3]:
manifest = load_pickle(MANIFEST_PATH)
assert isinstance(manifest, dict), 'Manifest must be a dictionary.'
assert manifest.get('schema_version') == CLUSTER_SCHEMA_VERSION, 'Unsupported manifest schema.'
assert len(manifest.get('splits', [])) == manifest.get('num_splits'), 'Manifest is incomplete.'

manifest_summary = {
    'schema_version': manifest['schema_version'],
    'num_splits': manifest['num_splits'],
    'split_fraction': manifest['split_fraction'],
    'base_seed': manifest['base_seed'],
    'number_of_diseases': len(manifest['disease_names']),
    'ppi_path_when_created': manifest['ppi_path'],
    'disease_path_when_created': manifest['disease_path'],
}
display(manifest_summary)

{'schema_version': 1,
 'num_splits': 2,
 'split_fraction': 0.75,
 'base_seed': 0,
 'number_of_diseases': 70,
 'ppi_path_when_created': '/home/fkp/vleeb/bioGraph/data/raw/PPI202207.txt',
 'disease_path_when_created': '/home/fkp/vleeb/bioGraph/data/raw/pcbi.1004120.s004.txt'}

In [4]:
inventory = []
expected_paths = []
for split_index in range(manifest['num_splits']):
    for group in METHOD_GROUPS:
        path = SHARD_ROOT / f'split_{split_index:03d}' / f'{group}.pkl'
        expected_paths.append(path)
        inventory.append({
            'split': split_index,
            'group': group,
            'exists': path.is_file(),
            'size_MiB': round(path.stat().st_size / 1024**2, 2) if path.is_file() else np.nan,
            'path': str(path.relative_to(PROJECT_ROOT)),
        })

display(inventory)
missing = [path for path in expected_paths if not path.is_file()]
assert not missing, 'Missing expected shards:\n' + '\n'.join(map(str, missing))

[{'split': 0,
  'group': 'classical',
  'exists': True,
  'size_MiB': 84.23,
  'path': 'outputs/results/shards/split_000/classical.pkl'},
 {'split': 0,
  'group': 'gcn',
  'exists': True,
  'size_MiB': 9.42,
  'path': 'outputs/results/shards/split_000/gcn.pkl'},
 {'split': 1,
  'group': 'classical',
  'exists': True,
  'size_MiB': 84.23,
  'path': 'outputs/results/shards/split_001/classical.pkl'},
 {'split': 1,
  'group': 'gcn',
  'exists': True,
  'size_MiB': 9.42,
  'path': 'outputs/results/shards/split_001/gcn.pkl'}]

## Structural and semantic validation

This loads each shard once, applies the project's artifact validator, checks method/score consistency, and compares every saved train/test partition with `splits.pkl`.

In [5]:
shards = {}
for split_index in range(manifest['num_splits']):
    for group in METHOD_GROUPS:
        path = SHARD_ROOT / f'split_{split_index:03d}' / f'{group}.pkl'
        shard = load_pickle(path)
        validate_benchmark_results(shard)
        shards[(split_index, group)] = shard

canonical_nodelist = list(shards[(0, 'classical')]['nodelist'])
canonical_nodes = set(canonical_nodelist)
inspection_rows = []
errors = []

for (split_index, group), shard in shards.items():
    nodelist = list(shard['nodelist'])
    methods = list(shard['config']['method_set'])
    expected_seed = manifest['splits'][split_index]['seed']
    expected_diseases = set(manifest['disease_names'])
    observed_diseases = [run['disease'] for run in shard['runs']]

    if len(nodelist) != len(set(nodelist)):
        errors.append(f'{split_index}/{group}: duplicate genes in nodelist')
    if set(nodelist) != canonical_nodes:
        errors.append(f'{split_index}/{group}: node set differs from the canonical node set')
    if len(observed_diseases) != len(set(observed_diseases)) or set(observed_diseases) != expected_diseases:
        errors.append(f'{split_index}/{group}: disease rows are missing or duplicated')

    split_matches = True
    scores_valid = True
    for run in shard['runs']:
        expected = manifest['splits'][split_index]['diseases'][run['disease']]
        if (run['seed'] != expected_seed or
                run['train_genes'] != expected['train_genes'] or
                run['test_genes'] != expected['test_genes']):
            split_matches = False
        if set(run['scores']) != set(methods):
            scores_valid = False
        for score in run['scores'].values():
            values = np.asarray(score)
            if values.shape != (len(nodelist),) or not np.isfinite(values).all():
                scores_valid = False

    if not split_matches:
        errors.append(f'{split_index}/{group}: seed or train/test genes disagree with splits.pkl')
    if not scores_valid:
        errors.append(f'{split_index}/{group}: invalid score keys, shape, or non-finite values')

    inspection_rows.append({
        'split': split_index, 'group': group, 'runs': len(shard['runs']),
        'nodes': len(nodelist), 'same_node_set': set(nodelist) == canonical_nodes,
        'same_node_order': nodelist == canonical_nodelist,
        'manifest_splits_match': split_matches, 'scores_valid': scores_valid,
        'methods': ', '.join(methods),
    })

display(inspection_rows)
assert not errors, 'Validation failed:\n- ' + '\n- '.join(errors)
print('All shards are complete and valid. Differing node order will be normalized during merge.')

[{'split': 0,
  'group': 'classical',
  'runs': 70,
  'nodes': 17504,
  'same_node_set': True,
  'same_node_order': True,
  'manifest_splits_match': True,
  'scores_valid': True,
  'methods': 'aNBR, rNBR, RWR, DK, DK*, QA0, QA1, QA*, DIAMOND'},
 {'split': 0,
  'group': 'gcn',
  'runs': 70,
  'nodes': 17504,
  'same_node_set': True,
  'same_node_order': False,
  'manifest_splits_match': True,
  'scores_valid': True,
  'methods': 'GCN'},
 {'split': 1,
  'group': 'classical',
  'runs': 70,
  'nodes': 17504,
  'same_node_set': True,
  'same_node_order': True,
  'manifest_splits_match': True,
  'scores_valid': True,
  'methods': 'aNBR, rNBR, RWR, DK, DK*, QA0, QA1, QA*, DIAMOND'},
 {'split': 1,
  'group': 'gcn',
  'runs': 70,
  'nodes': 17504,
  'same_node_set': True,
  'same_node_order': False,
  'manifest_splits_match': True,
  'scores_valid': True,
  'methods': 'GCN'}]

All shards are complete and valid. Differing node order will be normalized during merge.


In [6]:
# Hyperparameters and method lists must be stable within each method group.
for group in METHOD_GROUPS:
    reference = shards[(0, group)]['config']
    for split_index in range(1, manifest['num_splits']):
        candidate = shards[(split_index, group)]['config']
        assert candidate['method_set'] == reference['method_set'], f'{group} methods differ across splits.'
        assert candidate['hyperparameters'] == reference['hyperparameters'], f'{group} parameters differ across splits.'

method_table = [
    {'group': group,
     'methods': shards[(0, group)]['config']['method_set'],
     'hyperparameters': shards[(0, group)]['config']['hyperparameters']}
    for group in METHOD_GROUPS
]
display(method_table)

[{'group': 'classical',
  'methods': ['aNBR',
   'rNBR',
   'RWR',
   'DK',
   'DK*',
   'QA0',
   'QA1',
   'QA*',
   'DIAMOND'],
  'hyperparameters': {'rwr_return_prob': 0.4,
   'qa_t': 0.45,
   'beta': 0.5,
   'qa1_diag': 5.0,
   'qa_star_diag': 1.0,
   'dk_t': 0.3,
   'diamond_alpha': 9,
   'diamond_number_to_rank': 300}},
 {'group': 'gcn',
  'methods': ['GCN'],
  'hyperparameters': {'epochs': 100,
   'hidden_dim': 32,
   'disease_embedding_dim': 16,
   'learning_rate': 0.01,
   'weight_decay': 0.0001,
   'negative_ratio': 5,
   'inner_seed_fraction': 0.6666666666666666,
   'task_batch_size': 8}}]

## Align node order, merge, and save

The classical shard's node order is used as the canonical order. Each score array is reordered by matching its shard's gene IDs to that order. The result uses the same schema as the other files in `outputs/results`.

In [7]:
def aligned_scores(run: dict, shard_nodelist: list, target_nodelist: list) -> dict:
    if shard_nodelist == target_nodelist:
        return {method: np.asarray(values) for method, values in run['scores'].items()}
    source_position = {gene: position for position, gene in enumerate(shard_nodelist)}
    take = np.fromiter((source_position[gene] for gene in target_nodelist), dtype=np.intp)
    return {method: np.asarray(values)[take] for method, values in run['scores'].items()}


classical_methods = list(shards[(0, 'classical')]['config']['method_set'])
gcn_methods = list(shards[(0, 'gcn')]['config']['method_set'])
assert not (set(classical_methods) & set(gcn_methods)), 'Method names overlap across groups.'

combined_runs = []
for disease_name in manifest['disease_names']:
    for split_index, split_row in enumerate(manifest['splits']):
        combined_scores = {}
        for group in METHOD_GROUPS:
            shard = shards[(split_index, group)]
            matches = [run for run in shard['runs'] if run['disease'] == disease_name]
            assert len(matches) == 1
            combined_scores.update(aligned_scores(matches[0], list(shard['nodelist']), canonical_nodelist))
        expected = split_row['diseases'][disease_name]
        combined_runs.append({
            'disease': disease_name,
            'seed': split_row['seed'],
            'train_genes': expected['train_genes'],
            'test_genes': expected['test_genes'],
            'scores': combined_scores,
        })

combined = {
    'schema_version': ARTIFACT_SCHEMA_VERSION,
    'config': {
        'disease_set': manifest['disease_names'],
        'method_set': classical_methods + gcn_methods,
        'num_runs': manifest['num_splits'],
        'split_fraction': manifest['split_fraction'],
        'base_seed': manifest['base_seed'],
        'hyperparameters': {
            'classical': shards[(0, 'classical')]['config']['hyperparameters'],
            'gcn': shards[(0, 'gcn')]['config']['hyperparameters'],
        },
    },
    'nodelist': canonical_nodelist,
    'runs': combined_runs,
}

validate_benchmark_results(combined)
atomic_pickle_dump(combined, OUTPUT_PATH)
print(f'Wrote {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({OUTPUT_PATH.stat().st_size / 1024**2:.2f} MiB)')

Wrote outputs/results/results_methods_all_shards.pkl (187.10 MiB)


## Reload the single file and verify it

In [8]:
reloaded = load_pickle(OUTPUT_PATH)
validate_benchmark_results(reloaded)
assert reloaded['config'] == combined['config']
assert reloaded['nodelist'] == combined['nodelist']
assert len(reloaded['runs']) == len(manifest['disease_names']) * manifest['num_splits']
assert all(set(run['scores']) == set(reloaded['config']['method_set']) for run in reloaded['runs'])

final_summary = {
    'output_file': str(OUTPUT_PATH.relative_to(PROJECT_ROOT)),
    'file_size_MiB': round(OUTPUT_PATH.stat().st_size / 1024**2, 2),
    'nodes': len(reloaded['nodelist']),
    'diseases': len(reloaded['config']['disease_set']),
    'splits_per_disease': reloaded['config']['num_runs'],
    'total_run_rows': len(reloaded['runs']),
    'methods': ', '.join(reloaded['config']['method_set']),
}
display(final_summary)
print('Reload and validation succeeded.')

{'output_file': 'outputs/results/results_methods_all_shards.pkl',
 'file_size_MiB': 187.1,
 'nodes': 17504,
 'diseases': 70,
 'splits_per_disease': 2,
 'total_run_rows': 140,
 'methods': 'aNBR, rNBR, RWR, DK, DK*, QA0, QA1, QA*, DIAMOND, GCN'}

Reload and validation succeeded.
